In [ ]:
import scipy.io
import h5py

def load_mat(filename):
    try:
        return scipy.io.loadmat(filename)
    except NotImplementedError:
        return h5py.File(filename, "r")

data = load_mat("ACM.mat")


In [ ]:
from scipy.io import loadmat

data = loadmat("ACM.mat")
A = data["ACM"]

print(type(A))


KeyError: 'ACM'

In [ ]:
from scipy.io import loadmat

data = loadmat("ACM.mat")

print(data.keys())


dict_keys(['__header__', '__version__', '__globals__', 'TvsP', 'PvsA', 'PvsV', 'AvsF', 'VvsC', 'PvsL', 'PvsC', 'A', 'C', 'F', 'L', 'P', 'T', 'V', 'PvsT', 'CNormPvsA', 'RNormPvsA', 'CNormPvsC', 'RNormPvsC', 'CNormPvsT', 'RNormPvsT', 'CNormPvsV', 'RNormPvsV', 'CNormVvsC', 'RNormVvsC', 'CNormAvsF', 'RNormAvsF', 'CNormPvsL', 'RNormPvsL', 'stopwords', 'nPvsT', 'nT', 'CNormnPvsT', 'RNormnPvsT', 'nnPvsT', 'nnT', 'CNormnnPvsT', 'RNormnnPvsT', 'PvsP', 'CNormPvsP', 'RNormPvsP'])


In [ ]:
for k, v in data.items():
    if not k.startswith("__"):
        print(k, type(v), getattr(v, "shape", None))


TvsP <class 'scipy.sparse._coo.coo_matrix'> (1903, 12499)
PvsA <class 'scipy.sparse._coo.coo_matrix'> (12499, 17431)
PvsV <class 'scipy.sparse._coo.coo_matrix'> (12499, 196)
AvsF <class 'scipy.sparse._coo.coo_matrix'> (17431, 1804)
VvsC <class 'scipy.sparse._coo.coo_matrix'> (196, 14)
PvsL <class 'scipy.sparse._coo.coo_matrix'> (12499, 73)
PvsC <class 'scipy.sparse._coo.coo_matrix'> (12499, 14)
A <class 'numpy.ndarray'> (17431, 1)
C <class 'numpy.ndarray'> (14, 1)
F <class 'numpy.ndarray'> (1804, 1)
L <class 'numpy.ndarray'> (73, 1)
P <class 'numpy.ndarray'> (12499, 1)
T <class 'numpy.ndarray'> (1903, 1)
V <class 'numpy.ndarray'> (196, 1)
PvsT <class 'scipy.sparse._coo.coo_matrix'> (12499, 1903)
CNormPvsA <class 'scipy.sparse._coo.coo_matrix'> (12499, 17431)
RNormPvsA <class 'scipy.sparse._coo.coo_matrix'> (12499, 17431)
CNormPvsC <class 'scipy.sparse._coo.coo_matrix'> (12499, 14)
RNormPvsC <class 'scipy.sparse._coo.coo_matrix'> (12499, 14)
CNormPvsT <class 'scipy.sparse._coo.coo_matri

In [ ]:
pip install torch torch-geometric scipy h5py


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, HeteroData
from torch_geometric.nn import GCNConv, HANConv, HGTConv
from scipy.io import loadmat
from scipy.sparse import coo_matrix
import numpy as np
import random

# =========================
# CONFIG
# =========================
MODEL_NAME = "GCN"   # Choose: GCN | HAN | HGT
HIDDEN = 64
EPOCHS = 100
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Load ACM
# =========================
data_mat = loadmat("ACM.mat")

PvsT = data_mat["RNormPvsT"].tocsr()
PvsP = data_mat["PvsP"].tocoo()
PvsL = data_mat["PvsL"]

num_nodes = PvsT.shape[0]
num_features = PvsT.shape[1]
num_classes = PvsL.shape[1]

# Features
x = torch.tensor(PvsT.toarray(), dtype=torch.float)

# Labels
labels = PvsL.argmax(axis=1).A1
y = torch.tensor(labels, dtype=torch.long)

# Train/test split
idx = list(range(num_nodes))
random.shuffle(idx)
train_idx = torch.tensor(idx[:int(0.6*num_nodes)])
test_idx = torch.tensor(idx[int(0.6*num_nodes):])



class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(num_features, HIDDEN)
        self.conv2 = GCNConv(HIDDEN, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x


class HAN(nn.Module):
    def __init__(self, metadata):
        super().__init__()
        self.han = HANConv(
            in_channels=num_features,
            out_channels=HIDDEN,
            metadata=metadata,
            heads=4
        )
        self.lin = nn.Linear(HIDDEN, num_classes)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.han(x_dict, edge_index_dict)
        return self.lin(x_dict["paper"])


class HGT(nn.Module):
    def __init__(self, metadata):
        super().__init__()
        self.hgt = HGTConv(
            in_channels=num_features,
            out_channels=HIDDEN,
            metadata=metadata,
            heads=4
        )
        self.lin = nn.Linear(HIDDEN, num_classes)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.hgt(x_dict, edge_index_dict)
        return self.lin(x_dict["paper"])




if MODEL_NAME == "GCN":

    edge_index = torch.tensor(
        np.vstack((PvsP.row, PvsP.col)),
        dtype=torch.long
    )

    graph = Data(x=x, edge_index=edge_index, y=y)
    model = GCN().to(DEVICE)
    graph = graph.to(DEVICE)

elif MODEL_NAME in ["HAN", "HGT"]:

    PvsA = data_mat["PvsA"].tocoo()

    hetero = HeteroData()
    hetero["paper"].x = x
    hetero["paper"].y = y
    hetero["author"].x = torch.zeros((data_mat["PvsA"].shape[1], num_features))

    hetero["paper", "pa", "author"].edge_index = torch.tensor(
        np.vstack((PvsA.row, PvsA.col)), dtype=torch.long
    )
    hetero["author", "ap", "paper"].edge_index = torch.tensor(
        np.vstack((PvsA.col, PvsA.row)), dtype=torch.long
    )

    hetero = hetero.to(DEVICE)
    metadata = hetero.metadata()

    if MODEL_NAME == "HAN":
        model = HAN(metadata).to(DEVICE)
    else:
        model = HGT(metadata).to(DEVICE)

else:
    raise ValueError("Unknown model name")



optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    if MODEL_NAME == "GCN":
        out = model(graph.x, graph.edge_index)
        loss = F.cross_entropy(out[train_idx], graph.y[train_idx])
    else:
        out = model(hetero.x_dict, hetero.edge_index_dict)
        loss = F.cross_entropy(out[train_idx], hetero["paper"].y[train_idx])

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")



model.eval()
with torch.no_grad():
    if MODEL_NAME == "GCN":
        out = model(graph.x, graph.edge_index)
        pred = out.argmax(dim=1)
        acc = (pred[test_idx] == graph.y[test_idx]).float().mean()
    else:
        out = model(hetero.x_dict, hetero.edge_index_dict)
        pred = out.argmax(dim=1)
        acc = (pred[test_idx] == hetero["paper"].y[test_idx]).float().mean()

print(f"\nTest Accuracy ({MODEL_NAME}): {acc:.4f}")


Epoch 0, Loss: 4.2897
Epoch 10, Loss: 3.1282
Epoch 20, Loss: 2.7300
Epoch 30, Loss: 2.6317
Epoch 40, Loss: 2.5376
Epoch 50, Loss: 2.4348
Epoch 60, Loss: 2.3068
Epoch 70, Loss: 2.1747
Epoch 80, Loss: 2.0573
Epoch 90, Loss: 1.9605

Test Accuracy (GCN): 0.5106


In [ ]:
!pip install torch torch-geometric scipy scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, HeteroData
from torch_geometric.nn import GCNConv, HANConv, HGTConv
from scipy.io import loadmat
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import random

# =====================
# CONFIG
# =====================
MODEL_NAME = "ALL"  # GCN | HAN | HGT | ALL
HIDDEN = 64
EPOCHS = 100
LR = 0.005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# =====================
# LOAD DATA
# =====================
data_mat = loadmat("ACM.mat")

PvsT = data_mat["RNormPvsT"].tocsr()
PvsA = data_mat["PvsA"].tocsr()
PvsP = data_mat["PvsP"].tocoo()
PvsL = data_mat["PvsL"]

num_nodes = PvsT.shape[0]
num_features = PvsT.shape[1]
num_classes = PvsL.shape[1]

# Paper Features
x_paper = torch.tensor(PvsT.toarray(), dtype=torch.float)

# Labels
labels = PvsL.argmax(axis=1).A1
y_paper = torch.tensor(labels, dtype=torch.long)

# Train/Val/Test split
idx = np.random.permutation(num_nodes)
train_idx = torch.tensor(idx[:int(0.6*num_nodes)])
val_idx = torch.tensor(idx[int(0.6*num_nodes):int(0.8*num_nodes)])
test_idx = torch.tensor(idx[int(0.8*num_nodes):])

# =====================
# METRICS
# =====================
def evaluate(logits, labels, idx):
    preds = logits[idx].argmax(dim=1).cpu().numpy()
    true = labels[idx].cpu().numpy()
    acc = accuracy_score(true, preds)
    macro_f1 = f1_score(true, preds, average='macro')
    micro_f1 = f1_score(true, preds, average='micro')
    return acc, macro_f1, micro_f1

# =====================
# GCN
# =====================
class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(num_features, HIDDEN)
        self.conv2 = GCNConv(HIDDEN, num_classes)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

# =====================
# HAN (PAP + PTP)
# =====================
class HAN(nn.Module):
    def __init__(self, metadata):
        super().__init__()
        self.han = HANConv(
            in_channels=num_features,
            out_channels=HIDDEN,
            metadata=metadata,
            heads=4
        )
        self.lin = nn.Linear(HIDDEN, num_classes)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.han(x_dict, edge_index_dict)
        return self.lin(x_dict["paper"])

# =====================
# HGT (FULL GRAPH)
# =====================
class HGT(nn.Module):
    def __init__(self, metadata):
        super().__init__()
        self.hgt = HGTConv(
            in_channels=num_features,
            out_channels=HIDDEN,
            metadata=metadata,
            heads=4
        )
        self.lin = nn.Linear(HIDDEN, num_classes)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.hgt(x_dict, edge_index_dict)
        return self.lin(x_dict["paper"])

# =====================
# TRAIN FUNCTION
# =====================
def train_model(model, data, is_hetero=False):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()

        if not is_hetero:
            out = model(data.x, data.edge_index)
            loss = F.cross_entropy(out[train_idx], y_paper[train_idx])
        else:
            out = model(data.x_dict, data.edge_index_dict)
            loss = F.cross_entropy(out[train_idx], y_paper[train_idx])

        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        if not is_hetero:
            logits = model(data.x, data.edge_index)
        else:
            logits = model(data.x_dict, data.edge_index_dict)

    acc, macro_f1, micro_f1 = evaluate(logits, y_paper, test_idx)
    print(f"\nTest Accuracy: {acc:.4f}")
    print(f"Macro-F1: {macro_f1:.4f}")
    print(f"Micro-F1: {micro_f1:.4f}")
    print("="*50)

# =====================
# RUN MODELS
# =====================

# ----- GCN -----
if MODEL_NAME in ["GCN", "ALL"]:
    print("\nRunning GCN")
    edge_index = torch.tensor(
        np.vstack((PvsP.row, PvsP.col)),
        dtype=torch.long
    )
    graph = Data(x=x_paper, edge_index=edge_index).to(DEVICE)
    train_model(GCN(), graph, is_hetero=False)

# ----- HAN (PAP + PTP) -----
if MODEL_NAME in ["HAN", "ALL"]:
    print("\nRunning HAN (PAP + PTP)")

    PAP = (PvsA @ PvsA.T).tocoo()
    PTP = (PvsT @ PvsT.T).tocoo()

    hetero = HeteroData()
    hetero["paper"].x = x_paper
    hetero["paper", "PAP", "paper"].edge_index = torch.tensor(
        np.vstack((PAP.row, PAP.col)), dtype=torch.long
    )
    hetero["paper", "PTP", "paper"].edge_index = torch.tensor(
        np.vstack((PTP.row, PTP.col)), dtype=torch.long
    )

    hetero = hetero.to(DEVICE)
    metadata = hetero.metadata()

    train_model(HAN(metadata), hetero, is_hetero=True)

# ----- HGT (FULL GRAPH) -----
if MODEL_NAME in ["HGT", "ALL"]:
    print("\nRunning HGT (Full Heterogeneous Graph)")

    hetero = HeteroData()
    hetero["paper"].x = x_paper
    hetero["author"].x = torch.zeros((PvsA.shape[1], num_features))
    hetero["term"].x = torch.zeros((PvsT.shape[1], num_features))

    def add_edges(name, mat, src, dst):
        mat = mat.tocoo()
        hetero[(src, name, dst)].edge_index = torch.tensor(
            np.vstack((mat.row, mat.col)), dtype=torch.long
        )

    add_edges("pa", PvsA, "paper", "author")
    add_edges("ap", PvsA.T, "author", "paper")
    add_edges("pt", PvsT, "paper", "term")
    add_edges("tp", PvsT.T, "term", "paper")

    hetero = hetero.to(DEVICE)
    metadata = hetero.metadata()

    train_model(HGT(metadata), hetero, is_hetero=True)


OSError: could not read bytes

In [ ]:
# Faster install - use pre-built wheels
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric torch-scatter torch-sparse scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.datasets import IMDB
from sklearn.metrics import f1_score

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

def random_split(num_nodes, train_ratio=0.6, val_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(num_nodes)
    n_train = int(num_nodes * train_ratio)
    n_val = int(num_nodes * val_ratio)
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask[idx[:n_train]] = True
    val_mask[idx[n_train:n_train + n_val]] = True
    test_mask[idx[n_train + n_val:]] = True
    return train_mask, val_mask, test_mask

def preprocess_imdb(root="./data/IMDB"):
    dataset = IMDB(root=root)
    data = dataset[0]
    node_types = data.node_types

    num_nodes_type = {nt: data[nt].x.size(0) for nt in node_types}
    offsets = {}
    current = 0
    for nt in node_types:
        offsets[nt] = current
        current += num_nodes_type[nt]

    node_type_indices = {nt: torch.arange(offsets[nt], offsets[nt] + num_nodes_type[nt]) for nt in node_types}
    x_dict = {nt: data[nt].x.float() for nt in node_types}
    node_type_dims = {nt: x_dict[nt].size(1) for nt in node_types}

    edge_index_dict = {}
    relation_info = {}
    for (src_type, rel_name, dst_type) in data.edge_types:
        local_ei = data[(src_type, rel_name, dst_type)].edge_index
        rel_key = f"{src_type}__{rel_name}__{dst_type}"
        edge_index_dict[rel_key] = torch.stack([local_ei[0] + offsets[src_type], local_ei[1] + offsets[dst_type]])
        relation_info[rel_key] = (src_type, dst_type)

    labels = data["movie"].y.long()
    train_mask, val_mask, test_mask = random_split(labels.size(0))

    return {
        "x_dict": x_dict, "edge_index_dict": edge_index_dict, "node_type_indices": node_type_indices,
        "node_type_dims": node_type_dims, "relation_info": relation_info, "num_nodes": current,
        "labels": labels, "masks": {"train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask},
        "target_global_idx": node_type_indices["movie"]
    }

class HeteroHomogenizer(nn.Module):
    def __init__(self, node_type_dims, relation_info, num_nodes, hidden_dim=64, K=3):
        super().__init__()
        self.num_nodes, self.hidden_dim, self.K = num_nodes, hidden_dim, K
        self.relation_names = list(relation_info.keys())
        self.relation_info = relation_info
        self.projections = nn.ModuleDict({t: nn.Linear(d, hidden_dim) for t, d in node_type_dims.items()})
        self.alpha_raw = nn.ParameterDict({r: nn.Parameter(torch.zeros(K)) for r in self.relation_names})
        self.w_gate = nn.ParameterDict({r: nn.Parameter(torch.randn(hidden_dim) * 0.01) for r in self.relation_names})
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_rel = nn.ParameterDict({r: nn.Parameter(torch.randn(hidden_dim) * 0.01) for r in self.relation_names})
        self.mlp = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden_dim, hidden_dim))

    @staticmethod
    def sparse_neighbor_mean(X, edge_index):
        src, dst = edge_index
        N = X.size(0)
        deg = torch.zeros(N, device=X.device)
        deg.scatter_add_(0, dst, torch.ones_like(dst, dtype=X.dtype))
        deg_inv = torch.where(deg > 0, 1.0 / deg, torch.zeros_like(deg))
        out = torch.zeros_like(X)
        out.scatter_add_(0, dst.unsqueeze(1).expand(-1, X.size(1)), X[src])
        return out * deg_inv.unsqueeze(1)

    def forward(self, x_dict, edge_index_dict, node_type_indices):
        device = next(iter(x_dict.values())).device
        X = torch.zeros(self.num_nodes, self.hidden_dim, device=device)
        for t, idx in node_type_indices.items():
            X[idx] = self.projections[t](x_dict[t].to(device))

        p_raw = {}
        for r in self.relation_names:
            src_type, dst_type = self.relation_info[r]
            is_bipartite = (src_type != dst_type)
            ei = edge_index_dict[r].to(device)
            ei = torch.cat([ei, ei[[1, 0]]], dim=1)
            H = X
            powers = []
            for k in range(self.K):
                H = self.sparse_neighbor_mean(H, ei)
                if is_bipartite:
                    H = self.sparse_neighbor_mean(H, ei)
                powers.append(H)
            alpha = F.softmax(self.alpha_raw[r], dim=0)
            p_raw[r] = sum(alpha[k] * powers[k] for k in range(self.K))

        p_gated = {r: torch.sigmoid(p_raw[r] @ self.w_gate[r]).unsqueeze(1) * p_raw[r] for r in self.relation_names}
        p_normed = {r: p_gated[r] / p_gated[r].norm(dim=1, keepdim=True).clamp(min=1e-8) for r in self.relation_names}

        q = self.W_q(X)
        logits = torch.stack([(q * self.k_rel[r]).sum(dim=1) for r in self.relation_names], dim=1)
        beta = F.softmax(logits, dim=1)
        p_struct = sum(beta[:, i].unsqueeze(1) * p_normed[r] for i, r in enumerate(self.relation_names))

        src_all, dst_all, weight_all = [], [], []
        for i, r in enumerate(self.relation_names):
            s, t = edge_index_dict[r].to(device)
            src_all.extend([s, t])
            dst_all.extend([t, s])
            weight_all.extend([beta[s, i], beta[t, i]])

        src_cat, dst_cat, w_cat = torch.cat(src_all), torch.cat(dst_all), torch.cat(weight_all)
        w_cat *= (p_struct[src_cat] * p_struct[dst_cat]).sum(dim=1)
        A_homo = torch.sparse_coo_tensor(torch.stack([src_cat, dst_cat]), w_cat, (self.num_nodes, self.num_nodes)).coalesce()
        return self.mlp(torch.cat([X, p_struct], dim=1)), A_homo

class SimpleGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    @staticmethod
    def row_normalize_sparse(A):
        indices, values = A.indices(), A.values()
        deg = torch.zeros(A.size(0), device=values.device)
        deg.scatter_add_(0, indices[0], values.abs()).clamp_(min=1e-8)
        return torch.sparse_coo_tensor(indices, values / deg[indices[0]], A.size()).coalesce()

    def forward(self, X, A_sparse):
        A_norm = self.row_normalize_sparse(A_sparse)
        H = F.dropout(F.relu(self.lin1(torch.sparse.mm(A_norm, X))), p=self.dropout, training=self.training)
        return self.lin2(torch.sparse.mm(A_norm, H))

def train_and_evaluate(hidden_dim=64, K=3, epochs=50, lr=0.005, weight_decay=5e-4):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    processed = preprocess_imdb()
    x_dict = {k: v.to(device) for k, v in processed["x_dict"].items()}
    edge_index_dict = {k: v.to(device) for k, v in processed["edge_index_dict"].items()}
    node_type_indices = {k: v.to(device) for k, v in processed["node_type_indices"].items()}
    labels, masks = processed["labels"].to(device), {k: v.to(device) for k, v in processed["masks"].items()}
    target_idx = processed["target_global_idx"].to(device)
    num_classes = labels.max().item() + 1

    homogenizer = HeteroHomogenizer(processed["node_type_dims"], processed["relation_info"], processed["num_nodes"], hidden_dim, K).to(device)
    gcn = SimpleGCN(hidden_dim, hidden_dim, num_classes).to(device)
    optimizer = torch.optim.Adam(list(homogenizer.parameters()) + list(gcn.parameters()), lr=lr, weight_decay=weight_decay)

    def evaluate(split):
        homogenizer.eval()
        gcn.eval()
        with torch.no_grad():
            Z, A = homogenizer(x_dict, edge_index_dict, node_type_indices)
            preds = gcn(Z, A)[target_idx][masks[f"{split}_mask"]].argmax(dim=1)
            truth = labels[masks[f"{split}_mask"]]
            return f1_score(truth.cpu(), preds.cpu(), average="micro"), f1_score(truth.cpu(), preds.cpu(), average="macro")

    for epoch in range(1, epochs + 1):
        homogenizer.train()
        gcn.train()
        optimizer.zero_grad()
        Z, A = homogenizer(x_dict, edge_index_dict, node_type_indices)
        loss = F.cross_entropy(gcn(Z, A)[target_idx][masks["train_mask"]], labels[masks["train_mask"]])
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == 1:
            val_mi, val_ma = evaluate("val")
            print(f"Epoch {epoch:03d} | Loss {loss.item():.4f} | Val Micro-F1 {val_mi:.4f} | Val Macro-F1 {val_ma:.4f}")

    test_mi, test_ma = evaluate("test")
    print(f"\nTest Micro-F1: {test_mi:.4f} | Test Macro-F1: {test_ma:.4f}")

if __name__ == "__main__":
    train_and_evaluate()

  Preparing metadata (setup.py) ... done


Extracting data/IMDB/raw/IMDB_processed.zip
Processing...
Done!


Epoch 001 | Loss 1.1042 | Val Micro-F1 0.3637 | Val Macro-F1 0.1778
Epoch 010 | Loss 0.7144 | Val Micro-F1 0.5708 | Val Macro-F1 0.4491
Epoch 020 | Loss 0.3183 | Val Micro-F1 0.6632 | Val Macro-F1 0.6612
Epoch 030 | Loss 0.1117 | Val Micro-F1 0.6480 | Val Macro-F1 0.6480
Epoch 040 | Loss 0.0486 | Val Micro-F1 0.6281 | Val Macro-F1 0.6269
Epoch 050 | Loss 0.0326 | Val Micro-F1 0.6433 | Val Macro-F1 0.6423

Test Micro-F1: 0.6289 | Test Macro-F1: 0.6266


In [ ]:
# Install dependencies (run once in Colab)


import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.datasets import IMDB
from sklearn.metrics import f1_score

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def random_split(num_nodes, train_ratio=0.6, val_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(num_nodes)
    n_train = int(num_nodes * train_ratio)
    n_val = int(num_nodes * val_ratio)
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask[idx[:n_train]] = True
    val_mask[idx[n_train:n_train + n_val]] = True
    test_mask[idx[n_train + n_val:]] = True
    return train_mask, val_mask, test_mask

def preprocess_imdb(root="./data/IMDB"):
    dataset = IMDB(root=root)
    data = dataset[0]
    node_types = data.node_types
    edge_types = data.edge_types

    num_nodes_type = {nt: data[nt].x.size(0) for nt in node_types}
    offsets = {}
    current = 0
    for nt in node_types:
        offsets[nt] = current
        current += num_nodes_type[nt]

    node_type_indices = {nt: torch.arange(offsets[nt], offsets[nt] + num_nodes_type[nt], dtype=torch.long) for nt in node_types}
    x_dict = {nt: data[nt].x.float() for nt in node_types}
    node_type_dims = {nt: x_dict[nt].size(1) for nt in node_types}

    edge_index_dict = {}
    relation_info = {}
    for (src_type, rel_name, dst_type) in edge_types:
        local_ei = data[(src_type, rel_name, dst_type)].edge_index
        global_src = local_ei[0] + offsets[src_type]
        global_dst = local_ei[1] + offsets[dst_type]
        rel_key = f"{src_type}__{rel_name}__{dst_type}"
        edge_index_dict[rel_key] = torch.stack([global_src, global_dst], dim=0)
        relation_info[rel_key] = (src_type, dst_type)

    target_node_type = "movie"
    movie_data = data[target_node_type]
    labels = movie_data.y.long()

    if not hasattr(movie_data, "train_mask") or movie_data.train_mask is None:
        train_mask, val_mask, test_mask = random_split(labels.size(0), train_ratio=0.6, val_ratio=0.2, seed=42)
    else:
        train_mask, val_mask, test_mask = movie_data.train_mask, movie_data.val_mask, movie_data.test_mask

    return {
        "x_dict": x_dict,
        "edge_index_dict": edge_index_dict,
        "node_type_indices": node_type_indices,
        "node_type_dims": node_type_dims,
        "relation_info": relation_info,
        "num_nodes": current,
        "labels": labels,
        "masks": {"train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask},
        "target_node_type": target_node_type,
        "target_global_idx": node_type_indices[target_node_type],
    }

class HeteroHomogenizer(nn.Module):
    def __init__(self, node_type_dims, relation_info, num_nodes, hidden_dim=64, output_dim=64, K=3):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.K = K
        self.relation_names = list(relation_info.keys())
        self.relation_info = relation_info

        self.projections = nn.ModuleDict({t: nn.Linear(in_dim, hidden_dim) for t, in_dim in node_type_dims.items()})
        self.alpha_raw = nn.ParameterDict({r: nn.Parameter(torch.zeros(K)) for r in self.relation_names})
        self.w_gate = nn.ParameterDict({r: nn.Parameter(torch.randn(hidden_dim) * 0.01) for r in self.relation_names})
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_rel = nn.ParameterDict({r: nn.Parameter(torch.randn(hidden_dim) * 0.01) for r in self.relation_names})
        self.mlp = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden_dim, output_dim))

    @staticmethod
    def sparse_neighbor_mean(X, edge_index):
        src, dst = edge_index[0], edge_index[1]
        N, d = X.size()
        deg = torch.zeros(N, device=X.device, dtype=X.dtype)
        deg.scatter_add_(0, dst, torch.ones_like(dst, dtype=X.dtype))
        deg_inv = torch.zeros_like(deg)
        mask = deg > 0
        deg_inv[mask] = 1.0 / deg[mask]
        out = torch.zeros(N, d, device=X.device, dtype=X.dtype)
        out.scatter_add_(0, dst.unsqueeze(1).expand(-1, d), X[src])
        return out * deg_inv.unsqueeze(1)

    @staticmethod
    def symmetrize_edges(edge_index):
        rev = torch.stack([edge_index[1], edge_index[0]], dim=0)
        combined = torch.cat([edge_index, rev], dim=1)
        return torch.unique(combined, dim=1)

    def forward(self, x_dict, edge_index_dict, node_type_indices):
        device = next(iter(x_dict.values())).device
        N, d = self.num_nodes, self.hidden_dim
        X = torch.zeros(N, d, device=device)
        for t, idx in node_type_indices.items():
            X[idx] = self.projections[t](x_dict[t].to(device))

        p_raw = {}
        for r in self.relation_names:
            src_type, dst_type = self.relation_info[r]
            is_bipartite = (src_type != dst_type)
            ei = self.symmetrize_edges(edge_index_dict[r].to(device))
            H = X
            powers = []
            for k in range(self.K):
                if is_bipartite:
                    H = self.sparse_neighbor_mean(H, ei)
                    H = self.sparse_neighbor_mean(H, ei)
                else:
                    H = self.sparse_neighbor_mean(H, ei)
                powers.append(H)
            alpha = F.softmax(self.alpha_raw[r], dim=0)
            p_r = sum(alpha[k] * powers[k] for k in range(self.K))
            p_raw[r] = p_r

        p_gated = {r: torch.sigmoid(p_raw[r] @ self.w_gate[r]).unsqueeze(1) * p_raw[r] for r in self.relation_names}
        p_normed = {r: p_gated[r] / p_gated[r].norm(dim=1, keepdim=True).clamp(min=1e-8) for r in self.relation_names}

        q = self.W_q(X)
        logits = torch.stack([(q * self.k_rel[r]).sum(dim=1) for r in self.relation_names], dim=1)
        beta = F.softmax(logits, dim=1)
        p_struct = sum(beta[:, i].unsqueeze(1) * p_normed[r] for i, r in enumerate(self.relation_names))

        src_all, dst_all, weight_all = [], [], []
        for i, r in enumerate(self.relation_names):
            ei = edge_index_dict[r].to(device)
            s, t = ei[0], ei[1]
            src_all.extend([s, t])
            dst_all.extend([t, s])
            weight_all.extend([beta[s, i], beta[t, i]])

        src_cat, dst_cat = torch.cat(src_all), torch.cat(dst_all)
        w_cat = torch.cat(weight_all)
        sim = (p_struct[src_cat] * p_struct[dst_cat]).sum(dim=1)
        w_cat = w_cat * sim
        A_homo = torch.sparse_coo_tensor(torch.stack([src_cat, dst_cat]), w_cat, (N, N)).coalesce()
        Z = self.mlp(torch.cat([X, p_struct], dim=1))
        return Z, A_homo, beta

class SimpleGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    @staticmethod
    def row_normalize_sparse(A):
        indices, values = A.indices(), A.values()
        N, row = A.size(0), indices[0]
        deg = torch.zeros(N, dtype=values.dtype, device=values.device)
        deg.scatter_add_(0, row, values.abs())
        deg = deg.clamp(min=1e-8)
        norm_values = values / deg[row]
        return torch.sparse_coo_tensor(indices, norm_values, A.size()).coalesce()

    def forward(self, X, A_sparse):
        A_norm = self.row_normalize_sparse(A_sparse)
        H = torch.sparse.mm(A_norm, X)
        H = F.relu(self.lin1(H))
        H = F.dropout(H, p=self.dropout, training=self.training)
        H = torch.sparse.mm(A_norm, H)
        return self.lin2(H)

def train_and_evaluate_imdb(hidden_dim=64, K=3, epochs=50, lr=0.005, weight_decay=5e-4, device_str="cuda" if torch.cuda.is_available() else "cpu"):
    set_seed(42)
    device = torch.device(device_str)
    processed = preprocess_imdb()
    x_dict = {k: v.to(device) for k, v in processed["x_dict"].items()}
    edge_index_dict = {k: v.to(device) for k, v in processed["edge_index_dict"].items()}
    node_type_indices = {k: v.to(device) for k, v in processed["node_type_indices"].items()}
    labels = processed["labels"].to(device)
    masks = {k: v.to(device) for k, v in processed["masks"].items()}
    target_idx = processed["target_global_idx"].to(device)
    num_classes = labels.max().item() + 1

    homogenizer = HeteroHomogenizer(node_type_dims=processed["node_type_dims"], relation_info=processed["relation_info"], num_nodes=processed["num_nodes"], hidden_dim=hidden_dim, output_dim=hidden_dim, K=K).to(device)
    gcn = SimpleGCN(in_dim=hidden_dim, hidden_dim=hidden_dim, out_dim=num_classes, dropout=0.5).to(device)
    optimizer = torch.optim.Adam(list(homogenizer.parameters()) + list(gcn.parameters()), lr=lr, weight_decay=weight_decay)

    def evaluate(split):
        homogenizer.eval()
        gcn.eval()
        with torch.no_grad():
            Z, A_homo, _ = homogenizer(x_dict, edge_index_dict, node_type_indices)
            logits = gcn(Z, A_homo)[target_idx]
            mask = masks[f"{split}_mask"]
            preds = logits[mask].argmax(dim=1)
            truth = labels[mask]
            acc = (preds == truth).float().mean().item()
            micro_f1 = f1_score(truth.cpu(), preds.cpu(), average="micro")
            macro_f1 = f1_score(truth.cpu(), preds.cpu(), average="macro")
        return acc, micro_f1, macro_f1

    for epoch in range(1, epochs + 1):
        homogenizer.train()
        gcn.train()
        optimizer.zero_grad()
        Z, A_homo, _ = homogenizer(x_dict, edge_index_dict, node_type_indices)
        logits = gcn(Z, A_homo)[target_idx]
        loss = F.cross_entropy(logits[masks["train_mask"]], labels[masks["train_mask"]])
        loss.backward()
        optimizer.step()

        if epoch % 5 == 0 or epoch == 1:
            train_acc, train_mi, train_ma = evaluate("train")
            val_acc, val_mi, val_ma = evaluate("val")
            print(f"Epoch {epoch:03d} | Loss {loss.item():.4f} | Train Micro-F1 {train_mi:.4f} | Val Micro-F1 {val_mi:.4f} | Val Macro-F1 {val_ma:.4f}")

    test_acc, test_mi, test_ma = evaluate("test")
    print(f"\nTest Accuracy: {test_acc:.4f} | Test Micro-F1: {test_mi:.4f} | Test Macro-F1: {test_ma:.4f}")

if __name__ == "__main__":
    train_and_evaluate_imdb()